# 01 · Carga de datos — Créditos


## 1. Librerías y ubicación del proyecto
El notebook puede abrirse desde la carpeta raíz o desde `mlops_pipeline`. El código busca la base sin depender de la ruta personal del computador.

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
BASE = next((p for p in [cwd / "Base_de_datos.csv", cwd.parent / "Base_de_datos.csv"] if p.exists()), None)
if BASE is None:
    raise FileNotFoundError("Abre VS Code desde la carpeta principal del proyecto: no se encontró Base_de_datos.csv")
print("Archivo encontrado:", BASE.resolve())

Archivo encontrado: /mnt/data/ML_CREDITOS_MIGUEL/Base_de_datos.csv


## 2. Lectura y vista inicial
`head()` presenta las primeras filas, `shape` el tamaño y `info` los tipos. El CSV se preserva sin eliminar filas aquí.

In [2]:
df = pd.read_csv(BASE, encoding="utf-8-sig", low_memory=False)
print("Filas:", df.shape[0], " | Columnas:", df.shape[1])
display(df.head())
print("Columnas:", df.columns.tolist())
df.info()

Filas: 10763  | Columnas: 23


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,45664.611528,1852560.0,12,32,Empleado,3500000,1000000,128650,95.227787,795.0,0,2,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,45666.471389,3181080.0,6,34,Empleado,5000000,2000000,441817,95.227787,796.0,0,7,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,45652.786771,670200.0,5,34,Independiente,5000000,2000000,108632,95.227787,727.0,5,0,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,45630.597801,506806.8,2,25,Independiente,3000000,900000,199684,95.227787,864.0,8,1,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,45777.779132,999780.0,10,26,Empleado,2000000,600000,92509,95.227787,771.0,0,3,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


Columnas: ['tipo_credito', 'fecha_prestamo', 'capital_prestado', 'plazo_meses', 'edad_cliente', 'tipo_laboral', 'salario_cliente', 'total_otros_prestamos', 'cuota_pactada', 'puntaje', 'puntaje_datacredito', 'cant_creditosvigentes', 'huella_consulta', 'saldo_mora', 'saldo_total', 'saldo_principal', 'saldo_mora_codeudor', 'creditos_sectorFinanciero', 'creditos_sectorCooperativo', 'creditos_sectorReal', 'promedio_ingresos_datacredito', 'tendencia_ingresos', 'Pago_atiempo']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  float64
 2   capital_prestado               10763 non-null  float64
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int6

## 3. Validación de estructura y objetivo
La variable `Pago_atiempo` es binaria (1 pagó, 0 no pagó). Hay fuerte desbalance; se tendrá en cuenta durante evaluación.

In [3]:
required = ["Pago_atiempo", "fecha_prestamo", "salario_cliente", "edad_cliente"]
assert all(c in df.columns for c in required), "Faltan columnas obligatorias"
assert df["Pago_atiempo"].dropna().isin([0,1]).all(), "La variable objetivo contiene valores distintos a 0 y 1"
print("Distribución absoluta:")
display(df["Pago_atiempo"].value_counts(dropna=False).rename(index={0:"No pagó a tiempo", 1:"Pagó a tiempo"}))
print("Distribución porcentual:")
display((100*df["Pago_atiempo"].value_counts(normalize=True)).round(2))
print("Registros duplicados exactos:",df.duplicated().sum())
print("Celdas nulas:",int(df.isna().sum().sum()))

Distribución absoluta:


Pago_atiempo
Pagó a tiempo       10252
No pagó a tiempo      511
Name: count, dtype: int64

Distribución porcentual:


Pago_atiempo
1    95.25
0     4.75
Name: proportion, dtype: float64

Registros duplicados exactos: 0
Celdas nulas: 7175


## 4. Interpretación inicial
La fuente contiene 10.763 registros y 23 variables. El 95,25 % se registra como pago a tiempo y el 4,75 % como incumplimiento. Si un modelo siempre predijera pago puntual, acertaría la mayoría y aun así fallaría en todos los casos que más interesa detectar. Por eso utilizaremos métricas centradas en la clase minoritaria.

## 5. Siguiente etapa
Abre `comprension_eda.ipynb`: allí analizamos nulos, valores inesperados, limpieza, distribuciones y relaciones. Después convertiremos estas decisiones en el script reutilizable `ft_engineering.py`.